# Median‑of‑Means (MoM) k‑NN Non‑Local Median Filter

**Goal**: Robust, non‑local denoising that behaves like a median filter but
selects neighbors non‑locally (k‑nearest within a search window) and aggregates
using **Median‑of‑Means** (MoM) for outlier resistance with reduced variance.

**Why MoM?** Standard medians are robust but can be noisy; means are smooth but
non‑robust. Median‑of‑Means splits the k neighbors into `groups` buckets,
averages within each, then takes the median of those means — a strong, simple
robust estimator.

---
## What this notebook gives you
* GPU‑accelerated implementation using **CuPy** (falls back to NumPy on CPU).
* Two similarity modes:
1. **pixel** (fast): k‑NN by absolute intensity difference.
2. **patch** (robust): k‑NN by L2 distance between p×p patches (online top‑k, tiled).
* Memory‑aware **tiling**, safe for 16 GB VRAM.
* Interactive ROI preview with **ipywidgets**.
* Drop‑in functions you can reuse inside your pipelines.

---
## Notation
* `r` – search radius; window size is `(2r+1)×(2r+1)`.
* `p` – odd patch size (e.g. 3, 5, 7). Patch mode requires odd `p`.
* `k` – number of nearest neighbors (k ≤ window_area).
* `groups` – number of MoM buckets (must divide `k`).

---
## Typical tuning hints
* Start with `mode="pixel"`, `r=5..9`, `k≈25..49`, `groups=5`.
* For textured images or salt‑pepper noise, switch to `mode="patch"`,
use `p=5 or 7`, `r=7..12`, `k≈25..64`, `groups=5..8`.
* Larger `r` → better structure preservation but more compute.
* If VRAM is tight, reduce `tile_hw`, `k`, or switch to `mode="pixel"`.

---
## Failure points to watch (confidence)
* `k > (2r+1)^2` (95%) → invalid; must be ≤ window area.
* `k % groups != 0` (90%) → adjust `k` or `groups`.
* Even `p` in patch mode (85%) → misalignment; use odd `p`.
* VRAM OOM for large tiles with patch mode (70%) → reduce `tile_hw`, `k`, or `r`.
* Different dtypes (70%) → we cast to `float32` for distances.

In [1]:
from __future__ import annotations
import math, os
from dataclasses import dataclass
from typing import Tuple, Literal, Optional

import numpy as _np
try:
    import cupy as _cp
    CUPY_AVAILABLE = True
except Exception:
    _cp = None
    CUPY_AVAILABLE = False

from tifffile import imread, imwrite
from tqdm import tqdm

# Proper widgets import block
try:
    import ipywidgets as widgets
    from IPython.display import display
    _WIDGETS = True
except Exception:
    _WIDGETS = False

def get_xp(prefer_gpu: bool = True):
    """Return CuPy if available & requested, else NumPy."""
    if prefer_gpu and CUPY_AVAILABLE:
        return _cp
    return _np

def is_cupy_array(a) -> bool:
    return CUPY_AVAILABLE and isinstance(a, _cp.ndarray)

xp = get_xp(prefer_gpu=True)
print(f"Backend: {'CuPy (GPU)' if xp is _cp else 'NumPy (CPU)'}")


Backend: CuPy (GPU)


In [2]:
def sliding_window_view(a, window_shape: Tuple[int, int]):
    """Unified NumPy/CuPy sliding window view for 2D arrays."""
    if is_cupy_array(a):
        return _cp.lib.stride_tricks.sliding_window_view(a, window_shape)
    return _np.lib.stride_tricks.sliding_window_view(a, window_shape)

def reflect_pad(a, pad_hw: Tuple[int, int]):
    """Reflect-pad 2D array on both axes by (ph, pw)."""
    ph, pw = pad_hw
    if is_cupy_array(a):
        return _cp.pad(a, ((ph, ph), (pw, pw)), mode='reflect')
    return _np.pad(a, ((ph, ph), (pw, pw)), mode='reflect')


## I/O setup

- `DATA_ROOT` defaults to `/home/askiran/data/`
- Set `in_path` to your TIFF (2D or 3D stack).
- `use_slice`: pick one slice for quick preview (optional).


In [4]:
DATA_ROOT = "/home/askiran/data/Peri_1"  # adjust if needed
in_path   = os.path.join(DATA_ROOT, "Lowmag_1.4ups_recon0232.tif")  # change to your file
out_dir   = os.path.join(DATA_ROOT, "mom_knn_out")
os.makedirs(out_dir, exist_ok=True)

use_slice: Optional[int] = None  # e.g. 0 for preview on a 3D stack; None keeps 2D

raw = imread(in_path)
orig_dtype = raw.dtype

if raw.ndim == 3 and use_slice is not None:
    img = raw[use_slice]
else:
    img = raw if raw.ndim == 2 else raw[0]

img = _np.ascontiguousarray(img)
img_f32 = img.astype(_np.float32, copy=False)
print(f"Loaded {in_path} | shape={img_f32.shape} dtype={img_f32.dtype}")


Loaded /home/askiran/data/Peri_1/Lowmag_1.4ups_recon0232.tif | shape=(1014, 990) dtype=float32


## Parameters

- `mode`: `'pixel'` or `'patch'`
- `r`, `k`, `groups`: neighborhood & MoM parameters
- `p`: patch size (odd) for patch mode
- `tile_hw`: tile size to bound VRAM


In [5]:
@dataclass
class MoMParams:
    mode: Literal['pixel','patch'] = 'pixel'
    r: int = 7
    k: int = 25
    groups: int = 5
    p: int = 5        # odd, patch mode only
    tile_hw: int = 512
    prefer_gpu: bool = True

params = MoMParams()
print(params)

MoMParams(mode='pixel', r=7, k=25, groups=5, p=5, tile_hw=512, prefer_gpu=True)


## Core helpers

- `median_of_means`: split last axis into `groups`, mean per group, median across means.
- `sliding_window_view`: NumPy/CuPy unified API.
- `reflect_pad`: reflect padding for borders.


In [6]:
def median_of_means(vals, groups: int, axis: int = -1):
    """
    Robust aggregator: split 'axis' into (groups, n//groups),
    take mean over the 'n//groups' dim, then median over 'groups'.

    Fix: normalize negative 'axis' so that axis+1 is correct even for axis=-1.
    """
    # Pick array module
    xp_local = _cp if is_cupy_array(vals) else _np

    # Normalize axis (handle negatives)
    ndim = vals.ndim
    axis = axis if axis >= 0 else ndim + axis
    if not (0 <= axis < ndim):
        raise ValueError(f"axis out of range for array with ndim={ndim}: axis={axis}")

    # Defensive: groups >= 1
    g = int(groups)
    if g < 1:
        g = 1

    # Trim neighbor count so it's divisible by groups
    n = vals.shape[axis]
    m = (n // g) * g
    if m != n:
        slicer = [slice(None)] * ndim
        slicer[axis] = slice(0, m)
        vals = vals[tuple(slicer)]
        n = m

    # Reshape: replace the 'axis' dim n -> (g, n//g)
    newshape = vals.shape[:axis] + (g, n // g) + vals.shape[axis+1:]
    # Use method reshape to avoid cupy/numpy differences
    reshaped = vals.reshape(newshape)

    # Mean across the bucket-size dim (the one right after 'axis')
    means = xp_local.mean(reshaped, axis=axis + 1)

    # Median across the 'groups' dim (which now sits at 'axis')
    return xp_local.median(means, axis=axis)

## k-NN by intensity (fast, vectorized)

Algorithm: per-pixel window, absolute difference to center, take k smallest diffs, aggregate by MoM or median.


In [7]:
def knn_pixel_tile(img2d, r: int, k: int, groups: int, use_mom: bool = True):
    xp_local = _cp if is_cupy_array(img2d) else _np
    sw = 2 * r + 1
    padded = reflect_pad(img2d, (r, r))
    win = sliding_window_view(padded, (sw, sw))  # (H, W, sw, sw)
    center = win[..., r, r]
    diffs = xp_local.abs(win - center[..., None, None])

    H, W = center.shape
    diffs_f = diffs.reshape(H, W, sw * sw)
    win_f   = win.reshape(H, W, sw * sw)

    idx = xp_local.argpartition(diffs_f, kth=min(k-1, sw*sw-1), axis=-1)[..., :k]
    neigh_vals = xp_local.take_along_axis(win_f, idx, axis=-1)

    if use_mom:
        out = median_of_means(neigh_vals, groups=groups, axis=-1)
    else:
        out = xp_local.median(neigh_vals, axis=-1)
    return out


## Online top-k helper

Maintain best `k` distances/values without storing all candidates (used in patch mode).


In [8]:
def _online_topk_update(best_d, best_v, cand_d, cand_v):
    """
    In-place update of best_k tensors with candidate (distance, value).
    best_*: (H, W, k), cand_*: (H, W)
    """
    xp_local = _cp if is_cupy_array(best_d) else _np
    H, W, K = best_d.shape
    idx_max = xp_local.argmax(best_d, axis=-1)  # (H, W)
    rr = xp_local.arange(H)[:, None]
    cc = xp_local.arange(W)[None, :]
    worst = best_d[rr, cc, idx_max]
    mask = cand_d < worst
    best_d[rr, cc, idx_max] = xp_local.where(mask, cand_d, worst)
    best_v[rr, cc, idx_max] = xp_local.where(mask, cand_v, best_v[rr, cc, idx_max])


## k-NN by patch L2 (robust, tiled)

- Reflect-pad by `r + p//2`.
- Compare `p×p` patches over all offsets inside the search window.
- Keep online top-k neighbors by distance; aggregate neighbor center values by MoM.


In [9]:
def knn_patch_tiled(img2d, r: int, k: int, groups: int,
                    p: int = 5, tile_hw: int = 512, use_mom: bool = True,
                    prefer_gpu: bool = True):
    assert p % 2 == 1, "Patch size p must be odd."
    xp_local = _cp if is_cupy_array(img2d) else _np
    H, W = img2d.shape
    p2 = p // 2
    pad = r + p2
    img_pad = reflect_pad(img2d, (pad, pad))
    patches = sliding_window_view(img_pad, (p, p))  # (H+2r, W+2r, p, p)

    out = xp_local.empty((H, W), dtype=img2d.dtype)
    offs = [(dy, dx) for dy in range(-r, r+1) for dx in range(-r, r+1)]

    for y0 in tqdm(range(0, H, tile_hw), desc="patch-tiled", leave=False):
        y1 = min(H, y0 + tile_hw)
        for x0 in range(0, W, tile_hw):
            x1 = min(W, x0 + tile_hw)
            center_p = patches[r+y0:r+y1, r+x0:r+x1, :, :]
            th, tw = center_p.shape[:2]

            best_d = xp_local.full((th, tw, k), xp_local.inf, dtype=_np.float32)
            best_v = xp_local.zeros((th, tw, k), dtype=_np.float32)

            for (dy, dx) in offs:
                neigh_p = patches[r+dy+y0:r+dy+y1, r+dx+x0:r+dx+x1, :, :]
                d = center_p - neigh_p
                dist = xp_local.sum(d * d, axis=(-1, -2)).astype(_np.float32)
                neigh_val = neigh_p[..., p2, p2].astype(_np.float32)
                _online_topk_update(best_d, best_v, dist, neigh_val)

            out_tile = median_of_means(best_v, groups=groups, axis=-1) if use_mom else xp_local.median(best_v, axis=-1)
            out[y0:y1, x0:x1] = out_tile.astype(img2d.dtype, copy=False)

    return out


## High-level wrapper

Validates parameters, moves data to GPU if available, runs the selected mode, returns NumPy array for saving.


In [10]:
def mom_knn_filter(img2d_np: _np.ndarray,
                   mode: Literal['pixel','patch'] = 'pixel',
                   r: int = 7, k: int = 25, groups: int = 5,
                   p: int = 5, tile_hw: int = 512,
                   prefer_gpu: bool = True,
                   use_mom: bool = True):
    assert k <= (2*r + 1) * (2*r + 1), "k must be ≤ window area"
    if use_mom:
        assert k % max(1, groups) == 0, "k must be divisible by groups for MoM"
    backend = get_xp(prefer_gpu=prefer_gpu)
    a = backend.asarray(img2d_np, dtype=backend.float32)

    if mode == 'pixel':
        out = knn_pixel_tile(a, r=r, k=k, groups=groups, use_mom=use_mom)
    elif mode == 'patch':
        out = knn_patch_tiled(a, r=r, k=k, groups=groups, p=p, tile_hw=tile_hw,
                              use_mom=use_mom, prefer_gpu=prefer_gpu)
    else:
        raise ValueError("mode must be 'pixel' or 'patch'")

    if backend is _cp:
        out = _cp.asnumpy(out)
    return out


# Estimate noise level & pick a starting patch

Rules of thumb

If noise is `light` and features are small → p=3..5, r=7..9, k=25..49.

If noise is `punchy` (salt/pepper, speckle) or patterns repeat → bump to p=5..7, r=9..13, k=49..64.

In [11]:
# Cell — noise estimate (MAD) on ROI
import numpy as np

roi = img_f32[64:64+256, 64:64+256]
med = np.median(roi)
sigma_mad = 1.4826 * np.median(np.abs(roi - med))
print("sigma ~", float(sigma_mad))

# Heuristic seed:
p = 5 if sigma_mad < 15 else 7      # data_range dependent; adjust threshold to your bit depth
r = 9 if p == 5 else 11
k = 49                               # divisible by several groups
groups = 7                           # robust; ensure k % groups == 0


sigma ~ 5764.3486328125


## ROI preview & tuning

Run on a small crop to tune `r, k, groups, p` and pick mode. Saves a preview TIFF.


In [12]:
# Define a simple ROI to preview
ry0, rx0, rh, rw = 64, 64, 1000, 900  # adjust
roi_np = img_f32[ry0:ry0+rh, rx0:rx0+rw]

def _run_preview(mode='pixel', r=7, k=25, groups=5, p=5, prefer_gpu=True):
    import time
    t0 = time.time()
    out = mom_knn_filter(roi_np, mode=mode, r=r, k=k, groups=groups, p=p,
                         tile_hw=512, prefer_gpu=prefer_gpu, use_mom=True)
    dt = time.time() - t0
    print(f"mode={mode} r={r} k={k} groups={groups} p={p} | {dt:.2f}s")
    prev_path = os.path.join(out_dir, f"preview_{mode}_r{r}_k{k}_g{groups}_p{p}.tif")
    imwrite(prev_path, out.astype(_np.float32))
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(10, 5))
        ax[0].imshow(roi_np, cmap='gray'); ax[0].set_title('ROI input'); ax[0].axis('off')
        ax[1].imshow(out,    cmap='gray'); ax[1].set_title('MoM k-NN output'); ax[1].axis('off')
        plt.show()
    except Exception:
        pass
    return out

if _WIDGETS:
    ui = widgets.VBox([
        widgets.Dropdown(options=['pixel','patch'], value='pixel', description='mode'),
        widgets.IntSlider(value=7,  min=1,  max=15,  step=1, description='r'),
        widgets.IntSlider(value=25, min=1,  max=121, step=1, description='k'),
        widgets.IntSlider(value=5,  min=1,  max=16,  step=1, description='groups'),
        widgets.IntSlider(value=5,  min=3,  max=13,  step=2, description='p (odd)'),
        widgets.Checkbox(value=True, description='prefer_gpu'),
    ])
    out = widgets.interactive_output(
        lambda mode, r, k, groups, p, prefer_gpu: _run_preview(mode, r, k, groups, p, prefer_gpu),
        {'mode': ui.children[0], 'r': ui.children[1], 'k': ui.children[2],
         'groups': ui.children[3], 'p': ui.children[4], 'prefer_gpu': ui.children[5]}
    )
    display(ui, out)
else:
    _ = _run_preview()


Output()

In [13]:
# Ensure we’re comparing the same thing
print("raw shape:", getattr(raw, "shape", None), "dtype:", getattr(raw, "dtype", None))
print("img_f32 (preview input) shape:", img_f32.shape)

# Quick full-image (single 2D) check on the first full slice (not ROI)
first2d = raw[0] if raw.ndim == 3 else raw
test_out = mom_knn_filter(first2d.astype(_np.float32),
                          mode=params.mode, r=params.r, k=params.k,
                          groups=params.groups, p=params.p, tile_hw=params.tile_hw,
                          prefer_gpu=params.prefer_gpu, use_mom=True)
print("first2d shape:", first2d.shape, " -> out shape:", test_out.shape)
assert first2d.shape == test_out.shape, "Output shape mismatch (would appear as cropping)"


raw shape: (1014, 990) dtype: uint16
img_f32 (preview input) shape: (1014, 990)
first2d shape: (1014, 990)  -> out shape: (1014, 990)


In [37]:
# r=0 => window size 1 => output must equal input
test = mom_knn_filter(img_f32, mode='pixel', r=0, k=1, groups=1, prefer_gpu=True)
assert test.shape == img_f32.shape and _np.allclose(test, img_f32), "r=0 should be identity"

# First full slice (not ROI), confirm no cropping
ref2d = raw[0] if raw.ndim == 3 else raw
out2d = mom_knn_filter(ref2d.astype(_np.float32), mode=params.mode, r=params.r, k=params.k,
                       groups=params.groups, p=params.p, tile_hw=params.tile_hw,
                       prefer_gpu=params.prefer_gpu, use_mom=True)
assert out2d.shape == ref2d.shape, "shape mismatch → would appear cropped"
print("Sanity OK:", ref2d.shape, "->", out2d.shape)


Sanity OK: (1016, 1525) -> (1016, 1525)


## Full run

- 2D: single call
- 3D: set `process_3d=True` to process per-slice


In [44]:
def run_full(in_arr: _np.ndarray, params: MoMParams, process_3d: bool = False):
    if in_arr.ndim == 2 or not process_3d:
        res = mom_knn_filter(in_arr.astype(_np.float32),
                             mode=params.mode, r=params.r, k=params.k,
                             groups=params.groups, p=params.p, tile_hw=params.tile_hw,
                             prefer_gpu=params.prefer_gpu, use_mom=True)
        out_path = os.path.join(out_dir, f"MoM_{params.mode}_r{params.r}_k{params.k}_g{params.groups}_p{params.p}.tif")
        imwrite(out_path, res.astype(orig_dtype, copy=False))
        print("Saved:", out_path)
        return res
    else:
        Z = in_arr.shape[0]
        out_stack = _np.empty_like(in_arr)
        for z in tqdm(range(Z), desc='3D-slices'):
            out_stack[z] = mom_knn_filter(in_arr[z].astype(_np.float32),
                                          mode=params.mode, r=params.r, k=params.k,
                                          groups=params.groups, p=params.p, tile_hw=params.tile_hw,
                                          prefer_gpu=params.prefer_gpu, use_mom=True).astype(orig_dtype, copy=False)
        out_path = os.path.join(out_dir, f"MoM3D_{params.mode}_r{params.r}_k{params.k}_g{params.groups}_p{params.p}.tif")
        imwrite(out_path, out_stack)
        print("Saved:", out_path)
        return out_stack

# Example (comment out or edit path/params before running)
# result = run_full(raw, params, process_3d=(raw.ndim==3))


# 📌 Parameter Guide & Tuning Playbook for the MoM k-NN Non-Local Filter

This cell explains **every parameter** used by the notebook and how to tune them for quality, speed, and memory. It applies to both **pixel** and **patch** modes and to 2D images and 3D stacks (slice-wise).

---

## Core filtering parameters

### `mode ∈ {'pixel','patch'}`
- **Definition (in this code):** Chooses the neighbor similarity metric.
  - `pixel`: neighbors ranked by absolute intensity difference to the center pixel, `|I(y',x') − I(y,x)|`.
  - `patch`: neighbors ranked by SSD between `p×p` patches centered at `(y,x)` and `(y',x')`.
- **Where used:** Branch in `mom_knn_filter()` → `knn_pixel_tile()` or `knn_patch_tiled()`.
- **Effect:** `pixel` is much faster and lighter on VRAM; `patch` is far more robust on textured/structured content and heavier noise.
- **Tuning tips:**
  - Start with `pixel` to get a baseline and rough values for `r,k,groups`.
  - Switch to `patch` when you need better structure preservation or have repetitive textures / impulsive noise.

---

### `r` (search radius, integer ≥ 0)
- **Definition:** Size of the non-local search window is `(2r+1)×(2r+1)`.
- **Where used:** Determines offsets `(dy,dx) ∈ [-r,r]²` in both modes.
- **Effect:** Larger `r` explores farther neighborhoods → often better matches but **O((2r+1)²)** more compute.
- **Constraints:** None beyond performance; any `r` is valid (padding preserves full FoV).
- **Tuning tips:**
  - Typical: `r = 7–11`.  
  - If textures repeat over larger distances, push to `r = 11–13`.  
  - If edges soften, try lowering `r` or increasing `p` (patch mode).

---

### `k` (number of neighbors, integer ≥ 1)
- **Definition:** After scoring all candidates in the window, keep the **k** smallest-distance neighbors; aggregate only these **k** values.
- **Where used:** Top-k selection (`argpartition` in pixel mode; online top-k in patch mode).
- **Constraint:** `k ≤ (2r+1)²` (or `≤ (2r+1)² − 1` if you exclude the center).
- **Effect:** Higher `k` lowers variance (smoother) but risks crossing edges (over-smoothing).
- **MoM constraint:** For MoM, prefer `k % groups == 0` (the code trims extras otherwise).
- **Tuning tips (rule of thumb):**  
  - Set `k ≈ 20–40%` of window area.  
  - Examples:  
    - `r=7 → (2r+1)² = 225 → k ≈ 32–64`  
    - `r=9 → 361 → k ≈ 64–96`
  - If results look noisy → increase `k`. If edges are washed → decrease `k`.

---

### `groups` (number of MoM buckets, integer ≥ 1)
- **Definition:** Median-of-Means splits the `k` neighbor values into `groups` equal buckets, **mean** in each bucket → **median** across bucket means.
- **Where used:** `median_of_means(..., groups=groups)`.
- **Constraint:** Prefer `k % groups == 0`. If not, we trim to the nearest multiple.
- **Effect:** Larger `groups` increases robustness to outliers (median across more buckets) but can slightly increase variance.
- **Tuning tips:** Start at `5`; try `7–10` if you see outliers (salt-and-pepper, hot pixels). Ensure `k` is divisible.

---

### `p` (patch size, odd integer ≥ 1; patch mode only)
- **Definition:** Side length of square patch in SSD distance. Must be **odd** (ensures a unique center).
- **Where used:** `knn_patch_tiled()`; padding is `pad = r + p//2`.
- **Effect:** Larger `p` improves texture matching (more context) but costs ~`p²` per offset.
- **Special case:** `p = 1` makes patch mode equivalent to pixel mode (same neighbor ordering); use `pixel` for speed if `p=1`.
- **Tuning tips:**  
  - Fine detail / light noise → `p=3–5`  
  - Stronger noise / complex texture → `p=5–7` (occasionally 9–11 if VRAM permits)  
  - If edges smear at large `r`, try increasing `p` before increasing `k`.

---

### `tile_hw` (tile size for patch mode, integer ≥ 128)
- **Definition:** Processes the image in tiles of `tile_hw×tile_hw` to bound VRAM usage in **patch** mode.
- **Where used:** Tiling loops in `knn_patch_tiled()`.
- **Effect on memory (dominant terms per tile):**  
  - Top-k buffers: `best_d` + `best_v` ≈ `2 * tile_hw² * k * 4` bytes (float32).  
  - Working math: up to `tile_hw² * p² * 4` bytes per offset.
- **Tuning tips:**  
  - Start `tile_hw = 512`. If you see OOM → `384` or `256`.  
  - Larger tiles = fewer loops (faster) if VRAM allows.

---

### `prefer_gpu ∈ {True, False}`
- **Definition:** Choose **CuPy** (GPU) when available; else **NumPy** (CPU).
- **Where used:** `get_xp(prefer_gpu)` in `mom_knn_filter()`.
- **Effect:** Orders-of-magnitude speedup on GPU, especially in patch mode.
- **Tuning tip:** Keep `True`. If you debug on CPU, temporarily set `False`.

---

### `use_mom ∈ {True, False}`
- **Definition:** If `True`, aggregate with **Median-of-Means**; if `False`, use plain **median** over the `k` values.
- **Where used:** End of both pixel/patch paths.
- **Effect:**  
  - MoM: robust like median but often less noisy (means inside small buckets).  
  - Median: slightly more robust to extreme outliers but can look jittery.
- **Tuning tips:**  
  - Default `True`. If output looks slightly “blocky” in flat regions, try `False` (median), or increase `k` and keep MoM.

---

## Preview & I/O parameters

### `in_path`, `out_dir`
- **Definition:** Input image/stack and output folder for results.
- **Where used:** Load with `tifffile.imread`, save preview/full outputs with `imwrite`.
- **Tip:** Keep outputs separated (e.g., `mom_knn_out/`) to avoid opening previews by mistake.

### `use_slice ∈ {None, int}`
- **Definition:** If input is 3D and `use_slice` is an integer, the preview uses that single 2D slice.
- **Tip:** For full stack, set `use_slice = None` and use `run_full(..., process_3d=True)`.

### ROI preview: `ry0, rx0, rh, rw`
- **Definition:** Region of interest for fast tuning. The preview saves files like `preview_*.tif` (they are *intentionally cropped*).
- **Tip:** Use previews to decide parameters; then run the full image/stack.

---

## Constraints & failure guards

- `k ≤ (2r+1)²` (window area) — **must hold**.
- `k % groups == 0` recommended for MoM (else values are trimmed).
- `p` must be **odd** (patch mode).
- Shape preservation: reflect padding guarantees **no cropping**; output shape equals input.

---

## Optimization workflow (practical playbook)

1. **Start simple:**
   - `mode='pixel'`, `r=7–9`, `k=25–49`, `groups=5`.
   - Tune on a **small ROI** (preview cell).

2. **If structure/texture is important or noise is heavy**, switch to:
   - `mode='patch'`, `p=5` (or 7), `r=9–11`, `k=49–64`, `groups=7–8`.
   - If edges blur, try **higher `p`** (5→7) before raising `k` or `r`.

3. **Balance speed & memory (patch mode):**
   - If VRAM is tight → **reduce `tile_hw`** (512→384→256) or **lower `k`**.
   - Keep `prefer_gpu=True`.

4. **Fine-tune `k` & `groups` for MoM:**
   - Target `k ≈ 20–40%` of window area.
   - Pick `groups ∈ {5,7,8,10}` so `k % groups == 0`.
   - If impulsive outliers remain → increase `groups`.

5. **(Optional) Exclude center neighbor:**
   - Pixel mode: set center distance to `+∞` before top-k.
   - Patch mode: skip offset `(0,0)`.
   - This enforces strictly non-center neighbors.

6. **Verify & scale up:**
   - Sanity check one full 2D slice: input/output shapes must match.
   - Run `run_full(..., process_3d=True)` for the whole stack (slice-wise).

---

## Quick reference formulas

- **Window area:** `W = (2r+1)²`
- **Candidate patches (per pixel):** `W` (or `W−1` if center excluded)
- **Top-k memory (per tile):** `≈ 2 * tile_hw² * k * 4` bytes
- **Patch math (per offset):** `≈ tile_hw² * p² * 4` bytes
- **Heuristic `k`:** `k ≈ 0.2–0.4 * W`
- **Padding used by code:** `pad = r + p//2` (keeps full FoV)

---

## When to change what

- **Grainy / noisy but flat areas:** increase `k`, keep MoM on (`groups=7–8`).
- **Texture lost / edges softened:** decrease `k`, or increase `p` (patch), or lower `r`.
- **Speckle / salt-pepper outliers:** keep MoM, raise `groups` (and ensure `k % groups == 0`).
- **OOM or slow:** reduce `tile_hw`, or reduce `k`/`r`, or switch to `pixel` mode.

---

_Notes:_  
- `p=1` makes patch mode equivalent to pixel mode (same neighbor order), but pixel mode is **much faster** → route to `pixel` if `p==1`.  
- For extremely large images, you can wrap the full run with **Dask** chunking; the per-tile logic remains valid.

# Better grouping strategies (what & why)

### Distance-sorted round-robin (recommended default)
Sort the k neighbors by distance (most reliable first), then interleave them across groups so each bucket gets a mix of near & far items. This avoids putting all the “very close” neighbors into the same bucket and generally reduces variance of group means.

### Distance quantile (stratified) buckets
Sort by distance and split into equal-count quantiles (contiguous blocks). This ensures each bucket sits on a distinct reliability band (near → far). It’s simple and stable; a bit less variance-balanced than round-robin, but intuitive.

### Random partition (classic MoM and current above one)
Randomly shuffle the k neighbors (shared permutation) and chunk. This matches theory for MoM and avoids any order bias. Great as a quick baseline.

### Spatially stratified grouping (needs offsets)
Force diversity by assigning neighbors into rings (by radius) or quadrants (by angle) and then balance the groups from each ring/quadrant. This prevents one side of an edge from dominating a bucket.
• Pixel mode: you can decode (dy,dx) from the window index.
• Patch mode: store (dy,dx) when updating the top-k.

### Feature-cluster grouping (needs a tiny feature)
Build a small feature per neighbor—e.g., [distance, intensity], or [distance, gradient_mag], or a 2–4D PCA of the patch residuals—and k-means into groups. Take the mean per cluster, then median across clusters. This makes buckets “internally coherent.”

### Bootstrapped MoM (“bag MoM”)
Do B bootstrap draws of size k (or k′≤k), compute each subset’s mean, then median across the B means. Boosts stability and is very robust; cost scales with B.

### Trimmed/Winsorized inner mean
Inside each bucket, don’t use a raw mean: use a trimmed mean (drop α% lows/highs) or a Huber mean. Then keep the median across buckets. This reduces sensitivity to small within-bucket outliers.

### Weighted inner mean (distance-aware)
Inside each group, weight samples by a decreasing function of distance (e.g., 1/(ε+d)). Keep the final median across groups to preserve robustness. Good when “nearer is truly better.”

### Adaptive groups & k
Keep the bucket size in a nice range (e.g., 4–16). If a user picks (k, groups) with tiny buckets, auto-adjust groups so k // groups is reasonable (and divisible), or nudge k up/down to the closest multiple.